In [ ]:
!pip -q install -U huggingface_hub

from google.colab import userdata
from huggingface_hub import HfApi

# Recupera o token armazenado nos Secrets
token = userdata.get("HF_TOKEN")

# Testa a autenticação
api = HfApi(token=token)
usuario = api.whoami()

print("Conexão realizada com sucesso!")
print("Usuário:", usuario["name"])

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
Conexão realizada com sucesso!
Usuário: Cristiano-Logistica


In [ ]:
!pip install -q "huggingface_hub>=0.25.0,<2.0"

In [2]:
!pip install -q -U transformers accelerate

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

modelo_id = "Qwen/Qwen2.5-0.5B-Instruct"

print("Carregando o modelo...")

tokenizer = AutoTokenizer.from_pretrained(modelo_id)

modelo = AutoModelForCausalLM.from_pretrained(
    modelo_id,
    torch_dtype="auto",
    device_map="auto"
)

print("Modelo carregado!")
print("GPU disponível:", torch.cuda.is_available())

Carregando o modelo...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado!
GPU disponível: False


In [6]:
{
    "role": "system",
    "content": (
        "Você é o LogiAssist, um assistente de triagem de ocorrências logísticas. "
        "Responda em português brasileiro, com clareza e objetividade. "
        "Nunca invente informações, prazos, status, causas ou procedimentos. "
        "A ausência de atualização no rastreamento NÃO comprova extravio, "
        "independentemente da quantidade de dias sem movimentação. "
        "Não apresente hipóteses como fatos. "
        "Diferencie informações confirmadas, informações pendentes e hipóteses. "
        "Solicite os dados necessários para a análise, como transportadora, "
        "código de rastreamento, data da última atualização e último evento registrado. "
        "Oriente a validação com a transportadora ou responsável pela operação. "
        "Não confirme extravio sem evidência formal."
    )
}

{'role': 'system',
 'content': 'Você é o LogiAssist, um assistente de triagem de ocorrências logísticas. Responda em português brasileiro, com clareza e objetividade. Nunca invente informações, prazos, status, causas ou procedimentos. A ausência de atualização no rastreamento NÃO comprova extravio, independentemente da quantidade de dias sem movimentação. Não apresente hipóteses como fatos. Diferencie informações confirmadas, informações pendentes e hipóteses. Solicite os dados necessários para a análise, como transportadora, código de rastreamento, data da última atualização e último evento registrado. Oriente a validação com a transportadora ou responsável pela operação. Não confirme extravio sem evidência formal.'}

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

modelo_id = "Qwen/Qwen2.5-0.5B-Instruct"

print("Carregando o modelo...")

tokenizer = AutoTokenizer.from_pretrained(modelo_id)

modelo = AutoModelForCausalLM.from_pretrained(
    modelo_id,
    torch_dtype="auto",
    device_map="auto"
)

print("Modelo carregado!")
print("GPU disponível:", torch.cuda.is_available())

Carregando o modelo...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Modelo carregado!
GPU disponível: False


In [9]:
import requests
import json

url = "https://raw.githubusercontent.com/smcrstn/dio-lab-bia-do-futuro/main/data/procedimentos_logisticos.json"

resposta_json = requests.get(url)
resposta_json.raise_for_status()

base_conhecimento = resposta_json.json()

print("Base de conhecimento carregada!")
print("Categorias disponíveis:", len(base_conhecimento["ocorrencias"]))

Base de conhecimento carregada!
Categorias disponíveis: 7


In [10]:
print(json.dumps(
    base_conhecimento["ocorrencias"][0],
    ensure_ascii=False,
    indent=2
))

{
  "codigo": "ATRASO_ENTREGA",
  "categoria": "Atraso na entrega",
  "verificar": [
    "prazo previsto",
    "data de postagem ou coleta",
    "último evento e data",
    "ocorrências registradas"
  ],
  "acoes": [
    "Comparar a data atual com o prazo informado.",
    "Consultar o rastreamento no canal oficial.",
    "Registrar a consulta e encaminhar conforme procedimento interno."
  ],
  "nao_concluir": [
    "Prazo expirado não comprova extravio.",
    "Não prometer nova data sem confirmação oficial."
  ]
}


In [11]:
codigo_busca = "ATRASO_ENTREGA"

ocorrencia_encontrada = next(
    (
        item for item in base_conhecimento["ocorrencias"]
        if item["codigo"] == codigo_busca
    ),
    None
)

if ocorrencia_encontrada:
    print("Ocorrência encontrada:")
    print(json.dumps(
        ocorrencia_encontrada,
        ensure_ascii=False,
        indent=2
    ))
else:
    print("Ocorrência não encontrada.")

Ocorrência encontrada:
{
  "codigo": "ATRASO_ENTREGA",
  "categoria": "Atraso na entrega",
  "verificar": [
    "prazo previsto",
    "data de postagem ou coleta",
    "último evento e data",
    "ocorrências registradas"
  ],
  "acoes": [
    "Comparar a data atual com o prazo informado.",
    "Consultar o rastreamento no canal oficial.",
    "Registrar a consulta e encaminhar conforme procedimento interno."
  ],
  "nao_concluir": [
    "Prazo expirado não comprova extravio.",
    "Não prometer nova data sem confirmação oficial."
  ]
}


In [13]:
contexto_logistico = f"""
Categoria: {ocorrencia_encontrada['categoria']}

Informações que devem ser verificadas:
{json.dumps(ocorrencia_encontrada['verificar'], ensure_ascii=False)}

Ações recomendadas:
{json.dumps(ocorrencia_encontrada['acoes'], ensure_ascii=False)}

Conclusões proibidas:
{json.dumps(ocorrencia_encontrada['nao_concluir'], ensure_ascii=False)}
"""

print(contexto_logistico)


Categoria: Atraso na entrega

Informações que devem ser verificadas:
["prazo previsto", "data de postagem ou coleta", "último evento e data", "ocorrências registradas"]

Ações recomendadas:
["Comparar a data atual com o prazo informado.", "Consultar o rastreamento no canal oficial.", "Registrar a consulta e encaminhar conforme procedimento interno."]

Conclusões proibidas:
["Prazo expirado não comprova extravio.", "Não prometer nova data sem confirmação oficial."]



In [14]:
mensagens_integradas = [
    {
        "role": "system",
        "content": (
            "Você é o LogiAssist, assistente de triagem de operações logísticas. "
            "Responda em português brasileiro. "
            "Utilize os procedimentos fornecidos como referência principal. "
            "Não invente informações, prazos, status ou causas. "
            "Não conclua que houve extravio apenas pela ausência de atualização. "
            "Se faltarem dados, solicite as informações necessárias. "
            "Não prometa datas de entrega sem confirmação oficial.\n\n"
            "BASE DE CONHECIMENTO:\n"
            + contexto_logistico
        )
    },
    {
        "role": "user",
        "content": (
            "Um pedido está sem atualização no rastreamento há quatro dias. "
            "Isso confirma extravio? O que devo verificar e quais ações devo tomar?"
        )
    }
]

print("Mensagens preparadas para o LogiAssist!")

Mensagens preparadas para o LogiAssist!


In [15]:
texto_integrado = tokenizer.apply_chat_template(
    mensagens_integradas,
    tokenize=False,
    add_generation_prompt=True
)

entradas_integradas = tokenizer(
    texto_integrado,
    return_tensors="pt"
).to(modelo.device)

with torch.no_grad():
    saida_integrada = modelo.generate(
        **entradas_integradas,
        max_new_tokens=220,
        do_sample=False
    )

resposta_integrada = tokenizer.decode(
    saida_integrada[0][entradas_integradas["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("RESPOSTA DO LOGIASSIST:\n")
print(resposta_integrada)

RESPOSTA DO LOGIASSIST:

Sim, uma situação de extravio pode ser confirmada se o pedido estiver sem atualização no rastreamento por quatro dias. Para garantir que o problema seja resolvido, você deve seguir os procedimentos descritos abaixo:

1. **Verificar o prazo previsto**: Certifique-se de que o prazo previsto para o pedido ainda está disponível.
2. **Consultar o rastreamento no canal oficial**: Verifique se o pedido foi registrado corretamente e se há registros recentes.
3. **Registrar a consulta**: Realize a consulta no sistema de rastreamento para confirmar a situação.

Se o prazo for exato e o pedido ainda não foi atualizado, você deve encaminhar a consulta ao departamento de trânsito ou ao departamento de gestão de estoque. Caso contrário, pode considerar que o extravio ocorreu porque o pedido foi cancelado ou não foi enviado no momento certo.


In [18]:
def validar_ocorrencia(ocorrencia):
    regras = ocorrencia["nao_concluir"]

    return {
        "categoria": ocorrencia["categoria"],
        "verificacoes": ocorrencia["verificar"],
        "acoes": ocorrencia["acoes"],
        "restricoes": regras,
        "extravio_confirmado": False
    }


resultado_validado = validar_ocorrencia(ocorrencia_encontrada)

print(json.dumps(
    resultado_validado,
    ensure_ascii=False,
    indent=2
))

{
  "categoria": "Atraso na entrega",
  "verificacoes": [
    "prazo previsto",
    "data de postagem ou coleta",
    "último evento e data",
    "ocorrências registradas"
  ],
  "acoes": [
    "Comparar a data atual com o prazo informado.",
    "Consultar o rastreamento no canal oficial.",
    "Registrar a consulta e encaminhar conforme procedimento interno."
  ],
  "restricoes": [
    "Prazo expirado não comprova extravio.",
    "Não prometer nova data sem confirmação oficial."
  ],
  "extravio_confirmado": false
}


In [21]:
def gerar_resposta_segura(resultado):

    situacoes = {
        "Atraso na entrega":
            "É necessário verificar o prazo e o histórico antes de concluir a ocorrência.",

        "Rastreamento sem atualização":
            "A falta de atualização não comprova extravio.",

        "Tentativa de entrega sem sucesso":
            "É necessário verificar o motivo informado e a orientação oficial.",

        "Divergência de status":
            "Há necessidade de validar as fontes e os registros antes de confirmar o status.",

        "Avaria ou possível extravio":
            "A ocorrência precisa de apuração e preservação das evidências.",

        "Devolução":
            "É necessário identificar o motivo e acompanhar o retorno físico e sistêmico.",

        "Problema com endereço":
            "É necessário verificar os dados e as possibilidades de correção."
    }

    resposta = f"""
LOGIASSIST — TRIAGEM DE OCORRÊNCIA

Categoria: {resultado['categoria']}

Situação:
{situacoes.get(resultado['categoria'], 'Ocorrência pendente de análise.')}

Informações a verificar:
"""

    for item in resultado["verificacoes"]:
        resposta += f"- {item}\n"

    resposta += "\nAções recomendadas:\n"

    for acao in resultado["acoes"]:
        resposta += f"- {acao}\n"

    resposta += "\nRestrições:\n"

    for regra in resultado["restricoes"]:
        resposta += f"- {regra}\n"

    return resposta

In [22]:
for ocorrencia in base_conhecimento["ocorrencias"]:

    resultado = validar_ocorrencia(ocorrencia)

    resposta = gerar_resposta_segura(resultado)

    print("=" * 60)
    print(resposta)


LOGIASSIST — TRIAGEM DE OCORRÊNCIA

Categoria: Atraso na entrega

Situação:
É necessário verificar o prazo e o histórico antes de concluir a ocorrência.

Informações a verificar:
- prazo previsto
- data de postagem ou coleta
- último evento e data
- ocorrências registradas

Ações recomendadas:
- Comparar a data atual com o prazo informado.
- Consultar o rastreamento no canal oficial.
- Registrar a consulta e encaminhar conforme procedimento interno.

Restrições:
- Prazo expirado não comprova extravio.
- Não prometer nova data sem confirmação oficial.


LOGIASSIST — TRIAGEM DE OCORRÊNCIA

Categoria: Rastreamento sem atualização

Situação:
A falta de atualização não comprova extravio.

Informações a verificar:
- data e hora do último evento
- texto exato do status
- prazo previsto
- canal e horário da consulta

Ações recomendadas:
- Confirmar o código de rastreamento.
- Consultar o canal oficial.
- Comparar o último evento com o prazo previsto.
- Registrar e encaminhar se persistir a fa

In [23]:
ocorrencia_teste = {
    "categoria": "Rastreamento sem atualização",
    "codigo_rastreamento": "",
    "ultimo_evento": "",
    "data_ultimo_evento": "",
    "prazo_previsto": ""
}

print("TESTE DE INFORMAÇÕES INCOMPLETAS")
print("Categoria:", ocorrencia_teste["categoria"])

campos_obrigatorios = [
    "codigo_rastreamento",
    "ultimo_evento",
    "data_ultimo_evento",
    "prazo_previsto"
]

faltantes = [
    campo for campo in campos_obrigatorios
    if not ocorrencia_teste[campo]
]

print("\nInformações pendentes:")
for campo in faltantes:
    print("-", campo)

if faltantes:
    print("\nResultado: ocorrência pendente de informações.")
else:
    print("\nResultado: informações disponíveis para análise.")

TESTE DE INFORMAÇÕES INCOMPLETAS
Categoria: Rastreamento sem atualização

Informações pendentes:
- codigo_rastreamento
- ultimo_evento
- data_ultimo_evento
- prazo_previsto

Resultado: ocorrência pendente de informações.


In [24]:
categoria_teste = "Problema com pagamento"

ocorrencia_encontrada = next(
    (
        item for item in base_conhecimento["ocorrencias"]
        if item["categoria"].lower() == categoria_teste.lower()
    ),
    None
)

if ocorrencia_encontrada is None:
    print("Categoria não identificada na base de conhecimento.")
    print("Resultado: encaminhar para análise humana.")
else:
    print("Categoria identificada:", ocorrencia_encontrada["categoria"])

Categoria não identificada na base de conhecimento.
Resultado: encaminhar para análise humana.


In [25]:
total_categorias = len(base_conhecimento["ocorrencias"])
categorias_aprovadas = sum(
    1 for item in base_conhecimento["ocorrencias"]
    if item.get("categoria") and item.get("verificar")
    and item.get("acoes") and item.get("nao_concluir")
)

teste_incompleto_aprovado = len(faltantes) == 4
teste_categoria_desconhecida_aprovado = (
    ocorrencia_encontrada is None
)

print("AVALIAÇÃO DO LOGIASSIST")
print("-" * 40)

print(f"Categorias cadastradas: {total_categorias}")
print(f"Categorias com estrutura válida: {categorias_aprovadas}")
print(
    "Cobertura estrutural: "
    f"{categorias_aprovadas / total_categorias * 100:.1f}%"
)

print(
    "Teste de informações incompletas:",
    "APROVADO" if teste_incompleto_aprovado else "REVISAR"
)

print(
    "Teste de categoria desconhecida:",
    "APROVADO" if teste_categoria_desconhecida_aprovado else "REVISAR"
)

AVALIAÇÃO DO LOGIASSIST
----------------------------------------
Categorias cadastradas: 7
Categorias com estrutura válida: 7
Cobertura estrutural: 100.0%
Teste de informações incompletas: APROVADO
Teste de categoria desconhecida: APROVADO
